In [1]:
import numpy as np
import numpy.random as nprand
import math as math
import pandas as pd 

In [2]:
# Run in python console
import nltk; nltk.download('stopwords')

# Run in terminal or command prompt
#!python3 -m spacy download en

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wgo027\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
import re
from pprint import pprint

# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

# spacy for lemmatization
import spacy

# Plotting tools
import pyLDAvis
#import pyLDAvis.gensim  # don't skip this
import pyLDAvis.gensim_models as gensimvis
import matplotlib.pyplot as plt
%matplotlib inline

# Enable logging for gensim - optional
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

In [5]:
#Using Stop Words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['and', 'to', 'in', 'of', 'with', 'at', 'on', 'for', 'a', 'an', 'the', 'import', 'model', 'train', 'torch', 'january', 'jan', 'february', 'feb', 
                   'march', 'mar', 'april', 'apr', 'may', 'june', 'jun', 'july', 'jul', 'august', 'aug', 'september', 'sep', 'october', 'oct', 'november', 'nov', 
                   'december', 'dec'
   ])

In [6]:
# Load the data
df = pd.read_csv('../data/pyt_posts.csv')
df.head()

,Id,PostTypeId,ParentId,Title,Body,Score,Tags,ViewCount,FavoriteCount
0,34750268,1,NaN,Extracting the top-k value-indices from a 1-D ...,<p>Given a 1-D tensor in Torch (<code>torch.Te...,9,<python><lua><pytorch><torch>,12840.0,0.0
1,38543850,1,NaN,How to Display Custom Images in Tensorboard (e...,"<p>The <a href=""https://github.com/tensorflow/...",40,<python><tensorflow><matplotlib><pytorch><tens...,42232.0,0.0
2,41461670,1,NaN,cudnnRNNForwardTraining seqLength / xDesc usage,"<p>Let's say I have N sequences x[i], each wit...",7,<cudnn>,842.0,0.0
3,41767005,1,NaN,Python wheels: cp27mu not supported,"<p>I'm trying to install pytorch (<a href=""htt...",11,<python><linux><unicode><pytorch>,5358.0,0.0
4,41818618,1,NaN,PyTorch doesn't import after installing Anaconda,<p>I just installed PyTorch after installing A...,0,<macos><python-3.x><ipython><anaconda><pytorch>,1593.0,0.0


In [7]:
def sent_to_words(sentences):
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))  # deacc=True removes punctuations


#print(data_words[:10])

In [8]:
# Define functions for stopwords, bigrams, trigrams and lemmatization
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [14]:
for post in df['Id'].unique()[:]:
    print(post)
    for postType in df[df['Id'] == post]['PostTypeId'].unique():
        if postType == 1 or postType == 2:
            continue
        
        print(postType)

        data = df[(df['Id'] == post) & (df['PostTypeId'] == postType)]
        data_words = list(sent_to_words(data['Body']))

        # Build the bigram and trigram models
        bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
        trigram = gensim.models.Phrases(bigram[data_words], threshold=100)  

        # Faster way to get a sentence clubbed as a trigram/bigram
        bigram_mod = gensim.models.phrases.Phraser(bigram)
        trigram_mod = gensim.models.phrases.Phraser(trigram)

        # Remove Stop Words
        data_words_nostops = remove_stopwords(data_words)

        # Form Bigrams
        data_words_bigrams = make_bigrams(data_words_nostops)


        nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

        # Do lemmatization keeping only noun, adj, vb, adv
        data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

        # Create Dictionary
        id2word = corpora.Dictionary(data_lemmatized)

        # Create Corpus
        texts = data_lemmatized

        # Term Document Frequency
        corpus = [id2word.doc2bow(text) for text in texts]
        
        if len(corpus) < 1 or len(id2word) < 1:
            continue
        
        # Build LDA model
        lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                                id2word=id2word,
                                                num_topics=10, 
                                                random_state=100,
                                                update_every=1,
                                                chunksize=100,
                                                passes=10,
                                                alpha='auto',
                                                per_word_topics=True)

        # Print the Keyword in the 10 topics

        # Compute Perplexity
        print('\nPerplexity: ', lda_model.log_perplexity(corpus))  # a measure of how good the model is. lower the better.

        # Compute Coherence Score
        coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
        coherence_lda = coherence_model_lda.get_coherence()
        print('\nCoherence Score: ', coherence_lda)

        pyLDAvis.enable_notebook()
        vis = gensimvis.prepare(lda_model, corpus, id2word, mds='mmds')
        pyLDAvis.save_html(vis, f'{post}_{postType}.html')

34750268
38543850
41461670
41767005
41818618
41832779
41861354
41924453
42420709
42479902
42480111
42570645
42636323
42703500
42704283
42711144
42795226
42866654
42866752
42894882
42924324
42939708
42970009
42998216
43080583
43104252
43110987
43130124
43151642
43175778
43287476
43288486
43328632
43367075
43406693
43418825
43441673
43451125
43459013
43474072
43515400
43556339
43559536
43585263
43637591
43640454
43662364
43664444
43688282
43699517
43708693
43709859
43716525
43755077
43766386
43779500
43781526
43793168
43806326
43818246
43851657
43881353
43906242
43946813
43961051
43962599
43989310
44073705
44130851
44146655
44198736
44212831
44230829
44230907
44260217
44264018
44264443
44264962
44284506
44318616
44328530
44339375
44340848
44351134
44357055
44371560
44371682
44396749
44403886
44405297
44406819
44417500
44428784
44429199
44432978
44449728
44450435
44461772
44506380
44518677
44522661
44524901
44537895
44555372
44576035
44577501
44580450
44593141
44595338
44597523
44614977
4

C:\Users\wgo027\AppData\Roaming\Python\Python310\site-packages\sklearn\manifold\_mds.py:299: FutureWarning: The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.
  warnings.warn(


53398921
4

Perplexity:  -4.278037825697346

Coherence Score:  1.0
53401319
53401431
53402928
53403306
53403812
53404001
53405934
53410952
53416833
53419474
53421999
53424056
53437666
53438062
53438146
53440174
53442069
53442510
53445345
53447345
53447406
53448779
53449927
53452049
53453663
53454589
53455780
53455834
53461869
53462493
53463156
53463259
53463532
53464247
53464692
53465463
53465608
53467011
53467215
53467460
53467556
53471393
53471716
53474056
53474967
53475580
53475803
53476305
53477861
53479315
53479523
53491719
53494498
53496570
53500511
53500838
53503234
53505149
53507039
53507255
53507346
53512281
53515719
53523669
53530751
53530810
53531820
53532352
53538138
53539348
53541797
53542974
53544901
53546141
53547304
53548405
53549717
53559671
53562417
53564601
53568501
53569050
53569555
53570334
53570732
53571621
53576113
53577544
53577651
53578605
53580000
53580088
53581904
53584246
53586245
53588623
53593363
53596010
53600796
53605586
53605810
53607710
53612835
536144

C:\Users\wgo027\AppData\Roaming\Python\Python310\site-packages\sklearn\manifold\_mds.py:299: FutureWarning: The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.
  warnings.warn(



Perplexity:  -3.6307608957091966

Coherence Score:  1.0
53627246
4


C:\Users\wgo027\AppData\Roaming\Python\Python310\site-packages\sklearn\manifold\_mds.py:299: FutureWarning: The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.
  warnings.warn(



Perplexity:  -3.615363222360611

Coherence Score:  1.0
53628622
53628940
53631839
53631998
53639839
53641292
53644632
53652015
53653532
53660465
53660867
53663166
53665225
53667213
53671813
53672217
53673575
53674175
53678133
53682045
53683076
53683116
53684335
53686052
53688217
53689183
53689624
53691156
53692237
53693431
53693999
53695105
53696654
53697522
53697596
53698950
53699675
53706452
53706462
53709406
53710313
53711360
53711470
53711976
53715016
53716154
53716432
53721444
53721593
53725125
53726885
53732209
53732300
53735130
53735817
53736966
53740085
53742792
53743498
53744363
53745454
53751882
53752179
53759952
53763758
53768085
53768796
53769636
53769948
53775508
53781260
53784294
53784998
53788828
53791838
53793806
53794900
53796254
53797206
53797660
53798023
53800842
53802453
53803889
53807071
53810497
53813636
53814772
53818734
53819383
53820175
53826221
53828518
53831863
53832337
53835502
53836018
53837057
53841509
53841576
53843711
53844826
53852355
53853716
53858126

C:\Users\wgo027\AppData\Roaming\Python\Python310\site-packages\sklearn\manifold\_mds.py:299: FutureWarning: The default value of `normalized_stress` will change to `'auto'` in version 1.4. To suppress this warning, manually set the value of `normalized_stress`.
  warnings.warn(


54144867
54150684
54155275
54155969
54157468
54157839
54159814
54160505
54162500
54165651
54166206
54166865
54174054
54174854
54180152
54181381
54184807
54188885
54193209
54200785
54203451
54205116
54207204
54210286
54210933
54216920
54218604
54219691
54220042
54220249
54221434
54226309
54227444
54227872
54230231
54238519
54239125
54241512
54242123
54243064
54247143
54247715
54248646
54249399
54249737
54250651
54251007
54251798
54254313
54257503
54257695
54259943
54261531
54261604
54261892
54262455
54262689
54263155
54264338
54265661
54267919
54268029
54272565
54273680
54274089
54274716
54275436
54278148
54284077
54299965
54300136
54300707
54303890
54307225
54307824
54310861
54316020
54316053
54317378
54319419
54321678
54323427
54324844
54330778
54334829
54335125
54337853
54338081
54340330
54340766
54351299
54353327
54354465
54355067
54355310
54357836
54358280
54358292
54358669
54359243
54361763
54364457
54366020
54367644
54370810
54371840
54374935
54379214
54380140
54380830
54383474
5